In [1]:
import csv

def load_registrations(path):
    try:
        with open(path, 'r', newline='', encoding='utf-8') as f:
            data = list(csv.DictReader(f))
    except FileNotFoundError:
        print(f'File not found: {path}')
        return []
    else:
        print(len(data))
        return data
    finally:
        print('Load finished')

regs = load_registrations('fest_registrations.csv')
missing = load_registrations('missing.csv')

12
Load finished
File not found: missing.csv
Load finished


In [2]:
class InvalidTicketError(Exception):
    pass

def validate_row(row):
    ticket = row['ticket_type']
    amount = int(row['amount_inr'])
    if ticket not in ('Day', 'Pass'):
        raise InvalidTicketError(f'invalid ticket_type: {ticket}')
    if (ticket == 'Day' and amount != 150) or (ticket == 'Pass' and amount != 350):
        raise InvalidTicketError(f'invalid amount {amount} for {ticket}')
    return {'reg_id': row['reg_id'], 'name': row['name'], 'department': row['department'], 'ticket_type': ticket, 'amount_inr': amount}

test_rows = [
    {'reg_id': 'FEST001', 'name': 'aarav', 'department': 'CSE', 'ticket_type': 'Day', 'amount_inr': '150'},
    {'reg_id': 'FEST003', 'name': 'kabir', 'department': 'Civil', 'ticket_type': 'Pass', 'amount_inr': '350'},
    {'reg_id': 'FEST091', 'name': 'neha', 'department': 'EC', 'ticket_type': 'VIP', 'amount_inr': '500'},
    {'reg_id': 'FEST092', 'name': 'sam', 'department': 'Mech', 'ticket_type': 'Day', 'amount_inr': '350'},
    {'reg_id': 'FEST093', 'name': 'om', 'department': 'CSE', 'ticket_type': 'Pass', 'amount_inr': 'abc'}
]

for r in test_rows:
    try:
        print('valid:', validate_row(r))
    except (InvalidTicketError, ValueError) as e:
        print('error:', e)

valid: {'reg_id': 'FEST001', 'name': 'aarav', 'department': 'CSE', 'ticket_type': 'Day', 'amount_inr': 150}
valid: {'reg_id': 'FEST003', 'name': 'kabir', 'department': 'Civil', 'ticket_type': 'Pass', 'amount_inr': 350}
error: invalid ticket_type: VIP
error: invalid amount 350 for Day
error: invalid literal for int() with base 10: 'abc'


In [3]:
dirty_rows = [
    {'reg_id': 'FEST001', 'name': 'aarav', 'department': 'CSE', 'ticket_type': 'Day', 'amount_inr': '150'},
    {'reg_id': 'FEST002', 'name': 'diya', 'department': 'Mech', 'ticket_type': 'Day', 'amount_inr': '150'},
    {'reg_id': 'FEST003', 'name': 'kabir', 'department': 'Civil', 'ticket_type': 'VIP', 'amount_inr': '500'},
    {'reg_id': 'FEST004', 'name': 'ananya', 'department': 'EC', 'ticket_type': 'Day', 'amount_inr': '150'},
    {'reg_id': 'FEST005', 'name': 'rohan', 'department': 'CSE', 'ticket_type': 'Day', 'amount_inr': '350'},
    {'reg_id': 'FEST006', 'name': 'meera', 'department': 'Mech', 'ticket_type': 'Pass', 'amount_inr': '350'},
    {'reg_id': 'FEST007', 'name': 'dev', 'department': 'Civil', 'ticket_type': 'Day', 'amount_inr': '150'},
    {'reg_id': 'FEST008', 'name': 'kavya', 'department': 'EC', 'ticket_type': 'Pass', 'amount_inr': 'abc'},
    {'reg_id': 'FEST009', 'name': 'arjun', 'department': 'CSE', 'ticket_type': 'Pass', 'amount_inr': '350'},
    {'reg_id': 'FEST010', 'name': 'tara', 'department': 'Mech', 'ticket_type': 'Day', 'amount_inr': '150'}
]

fieldnames = ['reg_id', 'name', 'department', 'ticket_type', 'amount_inr']

with open('fest_dirty.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(dirty_rows)

loaded_dirty = load_registrations('fest_dirty.csv')
clean_rows = []
error_lines = []

try:
    for row in loaded_dirty:
        try:
            clean_rows.append(validate_row(row))
        except (InvalidTicketError, ValueError) as e:
            error_lines.append(f"{row['reg_id']}: {e}\n")
finally:
    print('Processing complete')

with open('fest_clean.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(clean_rows)

with open('fest_errors.txt', 'w', encoding='utf-8') as f:
    f.writelines(error_lines)

print(f'good={len(clean_rows)} bad={len(error_lines)}')

10
Load finished
Processing complete
good=7 bad=3
